# PCB component classifier: ViT-Small and MambaVision-T






In [ ]:
import torch

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. This notebook trains on CPU.")
print("ready")


In [ ]:
import os
import random
import time
import zipfile
import urllib.request
import xml.etree.ElementTree as ET
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 0
IMG_SIZE = 224
IOU_MATCH = 0.30
EPOCHS = 2
PATIENCE = 3
BATCH_SIZE = 16
LR = 2e-4
WEIGHT_DECAY = 0.05
MIN_TRAIN = 8
MIN_TEST = 2
TIMING_CROPS = 40
DATASET_URL = "https://ripl.cc.gatech.edu/data/pcb_wacv_2019.zip"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WORK = "/content/pcb_run" if IN_COLAB else os.path.join(os.getcwd(), "pcb_run")
os.makedirs(WORK, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
else:
    BATCH_SIZE = 8
    EPOCHS = 2


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print("work dir:", WORK)
if device.type == "cuda":
    print("device:", torch.cuda.get_device_name(0))
else:
    print("device: cpu  (batch size", BATCH_SIZE, ")")
    print("CPU fine-tuning is slow. A T4 GPU runtime is the practical way to finish both models.")


## Dataset



In [ ]:
def find_board_root(base):
    best, best_n = None, 0
    if not base or not os.path.isdir(base):
        return None
    for dirpath, dirnames, _filenames in os.walk(base):
        n = 0
        for name in dirnames:
            folder = os.path.join(dirpath, name)
            try:
                files = os.listdir(folder)
            except OSError:
                continue
            if any(f.endswith(".xml") for f in files):
                n += 1
        if n > best_n:
            best, best_n = dirpath, n
        if best_n >= 40:
            break
    return best if best_n >= 10 else None


def download_dataset(url, dest):
    print("downloading", url)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=180) as response, open(dest, "wb") as out:
        total = int(response.headers.get("Content-Length") or 0)
        done = 0
        next_report = 20 * 1024 * 1024
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            out.write(chunk)
            done += len(chunk)
            if done >= next_report:
                if total:
                    print(f"  {done / 1e6:.0f} / {total / 1e6:.0f} MB")
                else:
                    print(f"  {done / 1e6:.0f} MB")
                next_report += 20 * 1024 * 1024
    print("saved", dest, os.path.getsize(dest))


search_roots = [
    os.path.join(os.getcwd(), "data"),
    os.getcwd(),
    "/content/pcb_data",
    "/content",
]
DATA_ROOT = None
for root in search_roots:
    DATA_ROOT = find_board_root(root)
    if DATA_ROOT:
        break

if DATA_ROOT is None:
    data_home = "/content/pcb_data" if IN_COLAB else os.path.join(os.getcwd(), "data")
    os.makedirs(data_home, exist_ok=True)
    zip_path = os.path.join(data_home, "pcb_wacv_2019.zip")
    if not os.path.isfile(zip_path) or os.path.getsize(zip_path) < 50_000_000:
        download_dataset(DATASET_URL, zip_path)
    print("extracting")
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(data_home)
    DATA_ROOT = find_board_root(data_home)

if DATA_ROOT is None:
    raise FileNotFoundError("Could not find the PCB board folders after download.")

print("dataset:", DATA_ROOT)


## Labels



In [ ]:
SKIP_TYPES = {"text", "component text", "pads", "pins", "unknown"}
ALIASES = {
    "electrolytic capacitor": "capacitor",
    "zener diode": "diode",
    "resistor network": "resistor",
    "ferrite bead": "ferrite_bead",
    "emi filter": "emi_filter",
    "test point": "test_point",
}


def component_type(raw):
    raw = (raw or "").strip()
    if not raw:
        return None
    if raw.startswith('"'):
        end = raw.find('"', 1)
        typ = raw[1:end].strip().lower() if end > 1 else raw.lower()
    else:
        typ = raw.split()[0].lower()
    if typ in SKIP_TYPES or typ.startswith("component"):
        return None
    typ = ALIASES.get(typ, typ.replace(" ", "_"))
    if typ in SKIP_TYPES or typ.startswith("component"):
        return None
    return typ


def board_image(folder, filename):
    direct = os.path.join(folder, filename)
    if os.path.isfile(direct):
        return direct
    stem = os.path.splitext(filename)[0]
    images = []
    for name in os.listdir(folder):
        if os.path.splitext(name)[1].lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}:
            images.append(os.path.join(folder, name))
    for path in images:
        if os.path.splitext(os.path.basename(path))[0] == stem:
            return path
    if images:
        return images[0]
    raise FileNotFoundError(folder)


def load_ground_truth(xml_path):
    root = ET.parse(xml_path).getroot()
    width = int(root.find("size/width").text)
    height = int(root.find("size/height").text)
    filename = root.find("filename").text
    boxes = []
    for obj in root.findall("object"):
        typ = component_type(obj.find("name").text)
        if typ is None:
            continue
        box = obj.find("bndbox")
        x1 = int(float(box.find("xmin").text))
        y1 = int(float(box.find("ymin").text))
        x2 = int(float(box.find("xmax").text))
        y2 = int(float(box.find("ymax").text))
        if x2 - x1 < 4 or y2 - y1 < 4:
            continue
        boxes.append((x1, y1, x2, y2, typ))
    return width, height, filename, boxes


## 1. OpenCV finds the regions

On each assembled board the image is turned into masks. Contours on those masks become boxes. A box is kept when it has the size and aspect ratio of a part. The crop comes from the OpenCV box. The label comes from the part it matched.


In [ ]:
def iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    if inter <= 0:
        return 0.0
    area_a = max(1, (a[2] - a[0]) * (a[3] - a[1]))
    area_b = max(1, (b[2] - b[0]) * (b[3] - b[1]))
    return inter / (area_a + area_b - inter)


def nms(boxes, iou_thr=0.7, limit=4000):
    if not boxes:
        return []
    arr = np.asarray(boxes, dtype=np.float32)
    x1, y1, x2, y2 = arr.T
    areas = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    order = areas.argsort()[::-1]
    keep = []
    while order.size and len(keep) < limit:
        i = int(order[0])
        keep.append((int(x1[i]), int(y1[i]), int(x2[i]), int(y2[i])))
        if order.size == 1:
            break
        rest = order[1:]
        xx1 = np.maximum(x1[i], x1[rest])
        yy1 = np.maximum(y1[i], y1[rest])
        xx2 = np.minimum(x2[i], x2[rest])
        yy2 = np.minimum(y2[i], y2[rest])
        inter = np.maximum(0, xx2 - xx1) * np.maximum(0, yy2 - yy1)
        overlap = inter / (areas[i] + areas[rest] - inter + 1e-6)
        order = rest[overlap <= iou_thr]
    return keep


def propose_parts(bgr):
    """Threshold, find contours, and keep boxes that look like components."""
    height, width = bgr.shape[:2]
    long_side = max(height, width)
    scale = 1200 / long_side if long_side < 1200 else min(1.0, 1800 / long_side)
    if abs(scale - 1.0) < 1e-3:
        small = bgr
    else:
        interp = cv2.INTER_CUBIC if scale > 1 else cv2.INTER_AREA
        small = cv2.resize(bgr, (int(width * scale), int(height * scale)), interpolation=interp)

    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
    boxes = []
    small_area = gray.shape[0] * gray.shape[1]

    def add_mask(mask):
        contours, _hier = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            x, y, bw, bh = cv2.boundingRect(contour)
            if bw < 8 or bh < 8:
                continue
            area = bw * bh
            if area < 0.000015 * small_area or area > 0.12 * small_area:
                continue
            if max(bw / bh, bh / bw) > 15:
                continue
            boxes.append(
                (int(x / scale), int(y / scale), int((x + bw) / scale), int((y + bh) / scale))
            )

    blur = cv2.GaussianBlur(gray, (0, 0), 9)
    _thr, contrast = cv2.threshold(cv2.absdiff(gray, blur), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contrast = cv2.morphologyEx(contrast, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    add_mask(contrast)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (21, 21))
    for hat in (
        cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel),
        cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, kernel),
    ):
        _thr, mask = cv2.threshold(hat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
        add_mask(mask)

    _thr, saturation = cv2.threshold(hsv[:, :, 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    saturation = cv2.morphologyEx(saturation, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    add_mask(saturation)

    try:
        mser = cv2.MSER_create(5, 60, max(61, int(0.05 * small_area)))
        regions, _bboxes = mser.detectRegions(gray)
    except cv2.error:
        regions = []
    for points in regions:
        x, y, bw, bh = cv2.boundingRect(points)
        if bw < 8 or bh < 8:
            continue
        area = bw * bh
        if area < 0.000015 * small_area or area > 0.12 * small_area:
            continue
        if max(bw / bh, bh / bw) > 15:
            continue
        boxes.append((int(x / scale), int(y / scale), int((x + bw) / scale), int((y + bh) / scale)))

    clipped = []
    for x1, y1, x2, y2 in boxes:
        x1 = max(0, min(width - 1, x1))
        y1 = max(0, min(height - 1, y1))
        x2 = max(0, min(width, x2))
        y2 = max(0, min(height, y2))
        if x2 - x1 >= 6 and y2 - y1 >= 6:
            clipped.append((x1, y1, x2, y2))
    return nms(clipped, 0.7)


def pad_box(box, width, height, pad=0.04):
    x1, y1, x2, y2 = box
    bw = x2 - x1
    bh = y2 - y1
    x1 = max(0, int(x1 - pad * bw))
    y1 = max(0, int(y1 - pad * bh))
    x2 = min(width, int(x2 + pad * bw))
    y2 = min(height, int(y2 + pad * bh))
    return x1, y1, x2, y2


def resize_224(crop):
    interp = cv2.INTER_AREA if max(crop.shape[:2]) > IMG_SIZE else cv2.INTER_CUBIC
    return cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=interp)


def match_parts(gt_boxes, proposals):
    """One OpenCV box per labeled part, best IoU first."""
    pairs = []
    for g_i, gt in enumerate(gt_boxes):
        for p_i, prop in enumerate(proposals):
            score = iou(gt, prop)
            if score >= IOU_MATCH:
                pairs.append((score, g_i, p_i))
    pairs.sort(reverse=True)
    used_gt, used_prop = set(), set()
    matches = []
    for score, g_i, p_i in pairs:
        if g_i in used_gt or p_i in used_prop:
            continue
        used_gt.add(g_i)
        used_prop.add(p_i)
        matches.append((gt_boxes[g_i], proposals[p_i], score))
    return matches


## 2. Crop each box to 224×224

In [ ]:
CROP_DIR = os.path.join(WORK, "crops_224")
records = []
preview = None
if os.path.isdir(CROP_DIR):
    for label in sorted(os.listdir(CROP_DIR)):
        class_dir = os.path.join(CROP_DIR, label)
        if not os.path.isdir(class_dir):
            continue
        for name in sorted(os.listdir(class_dir)):
            if not name.lower().endswith(".jpg"):
                continue
            board = os.path.splitext(name)[0].rsplit("_", 1)[0]
            records.append({
                "path": os.path.join(class_dir, name),
                "label": label,
                "board": board,
                "iou": 1.0,
            })
if len(records) >= 500:
    sample_paths = [row["path"] for row in records[:8]]
    sample_img = cv2.cvtColor(cv2.imread(sample_paths[0]), cv2.COLOR_BGR2RGB)
    preview = {
        "title": f"reused {len(records)} saved 224x224 crops",
        "board": sample_img,
        "crops": [cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB) for p in sample_paths],
    }
    print(f"reusing {len(records)} crops from {CROP_DIR}")
else:
    if os.path.isdir(CROP_DIR):
        import shutil
        shutil.rmtree(CROP_DIR)
    os.makedirs(CROP_DIR, exist_ok=True)

board_names = sorted(
    name for name in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, name))
    and any(f.endswith(".xml") for f in os.listdir(os.path.join(DATA_ROOT, name)))
)

gt_total = 0
matched_total = 0

for index, board in enumerate([] if len(records) >= 500 else board_names, start=1):
    folder = os.path.join(DATA_ROOT, board)
    xml_path = next(os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".xml"))
    _w, _h, filename, gt_boxes = load_ground_truth(xml_path)
    image_path = board_image(folder, filename)
    bgr = cv2.imread(image_path)
    if bgr is None:
        print("skip unreadable", image_path)
        continue
    height, width = bgr.shape[:2]
    proposals = propose_parts(bgr)
    matches = match_parts(gt_boxes, proposals)
    gt_total += len(gt_boxes)
    matched_total += len(matches)
    print(
        f"[{index:02d}/{len(board_names)}] {board:24s} "
        f"gt={len(gt_boxes):4d} proposals={len(proposals):4d} "
        f"matched={len(matches):4d} recall={len(matches) / max(1, len(gt_boxes)):.2f}"
    )

    drawn = []
    for part_i, (gt, prop, score) in enumerate(matches):
        label = gt[4]
        x1, y1, x2, y2 = pad_box(prop, width, height)
        crop = bgr[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        crop = resize_224(crop)
        class_dir = os.path.join(CROP_DIR, label)
        os.makedirs(class_dir, exist_ok=True)
        crop_path = os.path.join(class_dir, f"{board}_{part_i:04d}.jpg")
        cv2.imwrite(crop_path, crop, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
        records.append({"path": crop_path, "label": label, "board": board, "iou": float(score)})
        drawn.append((x1, y1, x2, y2))

    if preview is None and drawn:
        vis = bgr.copy()
        for x1, y1, x2, y2 in drawn:
            cv2.rectangle(vis, (x1, y1), (x2, y2), (40, 220, 60), max(2, width // 400))
        show_scale = 900 / max(vis.shape[:2])
        vis = cv2.resize(vis, (int(width * show_scale), int(height * show_scale)), interpolation=cv2.INTER_AREA)
        sample_paths = [r["path"] for r in records if r["board"] == board][:8]
        preview = {
            "title": f"{board}: {len(drawn)} matched parts",
            "board": cv2.cvtColor(vis, cv2.COLOR_BGR2RGB),
            "crops": [cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB) for p in sample_paths],
        }

print(f"matched {matched_total} / {gt_total} parts  ({matched_total / max(1, gt_total):.1%})")
print("crops:", len(records))


In [ ]:
if not preview:
    raise RuntimeError("OpenCV matched no parts. Check that the dataset images and XML files extracted correctly.")

fig = plt.figure(figsize=(14, 8))
grid = fig.add_gridspec(2, 8, height_ratios=[3, 1], hspace=0.25)
ax = fig.add_subplot(grid[0, :])
ax.imshow(preview["board"])
ax.set_title(preview["title"])
ax.axis("off")
for i, crop in enumerate(preview["crops"]):
    ax_i = fig.add_subplot(grid[1, i])
    ax_i.imshow(crop)
    ax_i.axis("off")
plt.show()


## Same split for both models

Boards are shuffled once. About 70% of boards are train, 15% validation, and 15% test. Classes that are too rare to appear on both sides of the split are dropped. Both models then see this exact list.


In [ ]:
boards = sorted({r["board"] for r in records})
rng = random.Random(SEED)
rng.shuffle(boards)
n_test = max(1, int(round(0.15 * len(boards))))
n_val = max(1, int(round(0.15 * len(boards))))
test_boards = set(boards[:n_test])
val_boards = set(boards[n_test:n_test + n_val])
train_boards = set(boards[n_test + n_val:])

train_records = [r for r in records if r["board"] in train_boards]
val_records = [r for r in records if r["board"] in val_boards]
test_records = [r for r in records if r["board"] in test_boards]

train_counts = Counter(r["label"] for r in train_records)
test_counts = Counter(r["label"] for r in test_records)
keep = sorted(
    label for label in train_counts
    if train_counts[label] >= MIN_TRAIN and test_counts[label] >= MIN_TEST
)
label_to_y = {label: i for i, label in enumerate(keep)}
class_names = keep


def assign(rows):
    kept = []
    for row in rows:
        if row["label"] not in label_to_y:
            continue
        item = dict(row)
        item["y"] = label_to_y[row["label"]]
        kept.append(item)
    return kept


train_records = assign(train_records)
if device.type == "cpu":
    rng2 = random.Random(SEED)
    rng2.shuffle(train_records)
    capped = []
    seen = {}
    for row in train_records:
        seen[row["label"]] = seen.get(row["label"], 0) + 1
        if seen[row["label"]] <= 60:
            capped.append(row)
    train_records = capped
val_records = assign(val_records)
test_records = assign(test_records)
num_classes = len(class_names)

print(f"boards train/val/test: {len(train_boards)}/{len(val_boards)}/{len(test_boards)}")
print(f"crops train/val/test: {len(train_records)}/{len(val_records)}/{len(test_records)}")
print(f"classes ({num_classes}): {', '.join(class_names)}")
print("train counts:", dict(sorted(Counter(r["label"] for r in train_records).items())))


## 3. Fine-tune ViT-Small and MambaVision-T


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class CropSet(Dataset):
    def __init__(self, rows, transform):
        self.rows = rows
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        image = Image.open(row["path"]).convert("RGB")
        return self.transform(image), row["y"]


def as_logits(output):
    if isinstance(output, torch.Tensor):
        return output
    if isinstance(output, dict):
        return output["logits"]
    if hasattr(output, "logits"):
        return output.logits
    raise TypeError(type(output))


def make_loader(rows, transform, shuffle):
    generator = torch.Generator()
    generator.manual_seed(SEED)
    return DataLoader(
        CropSet(rows, transform),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0 if device.type == "cpu" or os.name == "nt" else 2,
        pin_memory=device.type == "cuda",
    )


@torch.no_grad()
def top1(model, loader):
    model.eval()
    correct = 0
    total = 0
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
            pred = as_logits(model(images)).argmax(1)
        correct += (pred == targets).sum().item()
        total += targets.numel()
    return correct / max(1, total)


def class_weights(rows):
    counts = Counter(r["y"] for r in rows)
    weights = torch.tensor(
        [1.0 / max(1, counts[i]) for i in range(num_classes)],
        dtype=torch.float32,
        device=device,
    )
    return weights / weights.sum() * num_classes


def build_vit(n_classes):
    import timm
    return timm.create_model("vit_small_patch16_224", pretrained=True, num_classes=n_classes)


def ensure_selective_scan():
    try:
        from mamba_ssm.ops.selective_scan_interface import selective_scan_fn  # noqa: F401
        return
    except Exception:
        pass
    import sys
    import types
    import torch.nn.functional as F

    def selective_scan_fn(u, delta, A, B, C, D=None, z=None, delta_bias=None, delta_softplus=False, return_last_state=False, **kwargs):
        dtype_in = u.dtype
        u = u.float()
        delta = delta.float()
        A = A.float()
        B = B.float()
        C = C.float()
        if delta_bias is not None:
            delta = delta + delta_bias.float().unsqueeze(-1)
        if delta_softplus:
            delta = F.softplus(delta)
        delta_a = torch.exp(torch.einsum("bdl,dn->bdln", delta, A))
        delta_b_u = torch.einsum("bdl,bnl,bdl->bdln", delta, B, u)
        state = u.new_zeros(u.shape[0], A.shape[0], A.shape[1])
        outputs = []
        for i in range(u.shape[2]):
            state = delta_a[:, :, i] * state + delta_b_u[:, :, i]
            outputs.append(torch.einsum("bdn,bn->bd", state, C[:, :, i]))
        y = torch.stack(outputs, dim=2)
        if D is not None:
            y = y + u * D.float().view(-1, 1)
        if z is not None:
            y = y * F.silu(z.float())
        y = y.to(dtype_in)
        if return_last_state:
            return y, state
        return y

    package = types.ModuleType("mamba_ssm")
    ops = types.ModuleType("mamba_ssm.ops")
    interface = types.ModuleType("mamba_ssm.ops.selective_scan_interface")
    interface.selective_scan_fn = selective_scan_fn
    package.ops = ops
    ops.selective_scan_interface = interface
    sys.modules["mamba_ssm"] = package
    sys.modules["mamba_ssm.ops"] = ops
    sys.modules["mamba_ssm.ops.selective_scan_interface"] = interface
    print("Using a PyTorch Mamba scan (no CUDA mamba-ssm).")


def build_mamba(n_classes):
    ensure_selective_scan()
    from transformers import AutoModelForImageClassification, PreTrainedModel

    if not hasattr(PreTrainedModel, "_mamba_tie_patch"):
        original = PreTrainedModel._move_missing_keys_from_meta_to_device

        def _move_missing_keys_from_meta_to_device(self, missing_keys, device_map, device_mesh, hf_quantizer):
            if not hasattr(self, "all_tied_weights_keys"):
                self.all_tied_weights_keys = {}
            return original(self, missing_keys, device_map, device_mesh, hf_quantizer)

        PreTrainedModel._move_missing_keys_from_meta_to_device = _move_missing_keys_from_meta_to_device
        PreTrainedModel._mamba_tie_patch = True

    model = AutoModelForImageClassification.from_pretrained(
        "nvidia/MambaVision-T-1K",
        trust_remote_code=True,
    )
    hidden = model.model.head.in_features
    model.model.head = nn.Linear(hidden, n_classes)
    return model


def train_model(name, builder):
    set_seed(SEED)
    print(f"\n===== {name} =====")
    ckpt_path = os.path.join(WORK, f"{name.replace(' ', '_').lower()}.pt")
    if os.path.isfile(ckpt_path):
        print("saved model found, skipping training:", ckpt_path)
        model = builder(num_classes).to(device)
        blob = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(blob["state_dict"])
        val_acc = top1(model, make_loader(val_records, eval_tf, shuffle=False))
        print(f"restored val_top1={val_acc:.4f}")
        return model, val_acc
    model = builder(num_classes).to(device)
    train_loader = make_loader(train_records, train_tf, shuffle=True)
    val_loader = make_loader(val_records, eval_tf, shuffle=False)
    criterion = nn.CrossEntropyLoss(weight=class_weights(train_records))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best_val = -1.0
    best_state = None
    stale = 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = 0.0
        seen = 0
        step = 0
        for images, targets in train_loader:
            step += 1
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device.type, enabled=use_amp):
                loss = criterion(as_logits(model(images)), targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            running += loss.item() * targets.numel()
            seen += targets.numel()
            if step == 1 or step % 10 == 0:
                print(f"  step {step} loss={running / max(1, seen):.4f}", flush=True)
        schedule.step()
        val_acc = top1(model, val_loader)
        print(f"epoch {epoch}/{EPOCHS}  loss={running / max(1, seen):.4f}  val_top1={val_acc:.4f}")
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print("early stop")
                break

    model.load_state_dict(best_state)
    model.to(device)
    ckpt = os.path.join(WORK, f"{name.replace(' ', '_').lower()}.pt")
    torch.save({"model": name, "state_dict": best_state, "classes": class_names}, ckpt)
    print("checkpoint:", ckpt)
    return model, best_val


@torch.no_grad()
def time_per_crop(model, tensors):
    model.eval()
    for i in range(10):
        batch = tensors[i % len(tensors)].unsqueeze(0).to(device, non_blocking=True)
        with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
            as_logits(model(batch))
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for tensor in tensors:
        batch = tensor.unsqueeze(0).to(device, non_blocking=True)
        with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
            as_logits(model(batch))
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - start) * 1000.0
    return elapsed_ms / len(tensors)


timing_tensors = []
for row in test_records[:TIMING_CROPS]:
    image = Image.open(row["path"]).convert("RGB")
    timing_tensors.append(eval_tf(image))
print("timing crops:", len(timing_tensors))


In [ ]:
test_loader = make_loader(test_records, eval_tf, shuffle=False)
results = []

for model_name, builder in (("ViT-Small", build_vit), ("MambaVision-T", build_mamba)):
    model, val_acc = train_model(model_name, builder)
    test_acc = top1(model, test_loader)
    ms = time_per_crop(model, timing_tensors)
    results.append({
        "model": model_name,
        "val_top1": val_acc,
        "test_top1": test_acc,
        "ms_per_crop": ms,
    })
    print(f"{model_name}: test top-1 {test_acc:.4f}   {ms:.2f} ms/crop")
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()


## See a detection

In [ ]:
# DEMO_DETECT
import timm

COLORS = [
    (255, 255, 0),
    (255, 0, 255),
    (0, 255, 255),
    (255, 128, 0),
    (180, 105, 255),
    (0, 215, 255),
    (147, 20, 255),
    (255, 191, 0),
]


def checkpoint_path():
    preferred = [
        os.path.join(WORK, "component_classifier.pt"),
        os.path.join(WORK, "mambavision-t.pt"),
        os.path.join(WORK, "vit-small.pt"),
    ]
    for path in preferred:
        if os.path.isfile(path):
            return path
    raise FileNotFoundError("No saved model in pcb_run. Run the training cell once.")


def load_saved_classifier():
    path = checkpoint_path()
    blob = torch.load(path, map_location=device, weights_only=False)
    names = blob["classes"]
    if "mamba" in blob["model"].lower():
        model = build_mamba(len(names))
    else:
        model = timm.create_model(
            "vit_small_patch16_224", pretrained=False, num_classes=len(names)
        )
    model.load_state_dict(blob["state_dict"])
    model.to(device).eval()
    stable = os.path.join(WORK, "component_classifier.pt")
    if os.path.abspath(path) != os.path.abspath(stable):
        torch.save(blob, stable)
    print("loaded", blob["model"], "from", path)
    return model, names


def busiest_board(names):
    best_name, best_n = None, -1
    for name in names:
        folder = os.path.join(DATA_ROOT, name)
        xml_path = next(
            os.path.join(folder, file_name)
            for file_name in os.listdir(folder)
            if file_name.endswith(".xml")
        )
        _w, _h, _fn, boxes = load_ground_truth(xml_path)
        if len(boxes) > best_n:
            best_name, best_n = name, len(boxes)
    return best_name


demo_model, demo_classes = load_saved_classifier()
demo_name = busiest_board(sorted(test_boards))
folder = os.path.join(DATA_ROOT, demo_name)
xml_path = next(os.path.join(folder, name) for name in os.listdir(folder) if name.endswith(".xml"))
_w, _h, filename, gt_boxes = load_ground_truth(xml_path)
bgr = cv2.imread(board_image(folder, filename))
height, width = bgr.shape[:2]
matches = match_parts(gt_boxes, propose_parts(bgr))

crops = []
boxes = []
for _gt, prop, _score in matches:
    x1, y1, x2, y2 = pad_box(prop, width, height)
    crop = bgr[y1:y2, x1:x2]
    if crop.size == 0:
        continue
    crops.append(Image.fromarray(cv2.cvtColor(resize_224(crop), cv2.COLOR_BGR2RGB)))
    boxes.append((x1, y1, x2, y2))

predicted = []
confidence = []
with torch.no_grad():
    for start in range(0, len(crops), 16):
        batch = torch.stack([eval_tf(image) for image in crops[start:start + 16]]).to(device)
        probs = torch.softmax(as_logits(demo_model(batch)), dim=1)
        conf, index = probs.max(dim=1)
        predicted.extend(demo_classes[i] for i in index.tolist())
        confidence.extend(float(value) for value in conf.tolist())

color_for = {name: COLORS[i % len(COLORS)] for i, name in enumerate(demo_classes)}
annotated = bgr.copy()
for (x1, y1, x2, y2), pred, conf in zip(boxes, predicted, confidence):
    color = color_for[pred]
    cv2.rectangle(annotated, (x1, y1), (x2, y2), color, max(2, width // 700))
    label = f"{pred} {conf:.2f}"
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = max(0.45, width / 3500)
    thickness = max(1, width // 1800)
    (tw, th), _ = cv2.getTextSize(label, font, scale, thickness)
    ty = y1 - 4 if y1 - th - 6 > 0 else y2 + th + 4
    cv2.putText(annotated, label, (x1, ty), font, scale, color, thickness, cv2.LINE_AA)

xs1 = [box[0] for box in boxes]
ys1 = [box[1] for box in boxes]
xs2 = [box[2] for box in boxes]
ys2 = [box[3] for box in boxes]
pad = 40
view = annotated[
    max(0, min(ys1) - pad):min(height, max(ys2) + pad),
    max(0, min(xs1) - pad):min(width, max(xs2) + pad),
]
long_side = max(view.shape[:2])
if long_side > 1400:
    scale = 1400 / long_side
    view = cv2.resize(view, (int(view.shape[1] * scale), int(view.shape[0] * scale)), interpolation=cv2.INTER_AREA)

out_dir = os.path.join(WORK, "detect")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, demo_name + ".jpg")
cv2.imwrite(out_path, view, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
print("Results saved to", out_path)

plt.figure(figsize=(14, 10))
plt.imshow(cv2.cvtColor(view, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.tight_layout()
plt.show()
